# 04 — UPDATE / DELETE без WHERE

> **`vuln_class`:** `DML_NO_WHERE` · **Риск:** 9/10 · **CWE-1284**

Запрос модификации данных без условия `WHERE` (или с предикатом, который всегда истинный — `WHERE 1=1`) затрагивает **все строки** таблицы. Часто — результат опечатки или копипасты, в проде сразу становится инцидентом.


## 🧒 Аналогия для ребёнка

У тебя в комнате стоят 1000 коробок. Мама говорит:
«**Покрась коробку #42 в красный**». Это правильное задание — ты
находишь коробку 42 и красишь её.

А теперь представь, что мама забыла назвать номер: «**Покрась коробку**».
Какую? Все? Ты в растерянности — и красишь **все 1000 коробок**.
Восстановить старый цвет уже нельзя.

В SQL то же самое: `UPDATE clients SET balance = 0` — где у тебя
нет `WHERE`, БД применит к **всей таблице**. У всех клиентов
баланс станет 0. Восстановить — только из бэкапа.


## 1. Setup — таблица клиентов с балансами


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief Создаёт мок-БД клиентов с балансами.
def setup_clients_db():
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE clients (
            client_id INTEGER PRIMARY KEY,
            full_name TEXT,
            balance   REAL
        )""")
    cur.executemany(
        "INSERT INTO clients (full_name, balance) VALUES (?, ?)",
        [("Иван И.", 1500.50), ("Мария П.", 87000.00), ("Олег С.", 250.00),
         ("Анна Р.", 1_000_000.00), ("Виктор М.", 0.50)],
    )
    conn.commit()
    return conn


conn = setup_clients_db()
section("Состояние таблицы clients ДО запроса")
show_result(conn.execute("SELECT * FROM clients").fetchall())


## 2. Опасный запрос — задумано обнулить одного клиента, забыли WHERE


In [ ]:
##
# @brief УЯЗВИМАЯ функция: разработчик хотел обнулить баланс одного клиента.
# @warning Забыли `WHERE client_id = ...` — обнулится ВСЯ таблица.
def zero_balance_BAD(conn, client_id: int):
    sql = "UPDATE clients SET balance = 0"  # ← АНТИПАТТЕРН: нет WHERE
    print(f"  Выполняется: {sql}")
    conn.execute(sql)
    conn.commit()


section("УЯЗВИМЫЙ вызов: думали обнулить #3, но...")
zero_balance_BAD(conn, 3)

section("Состояние таблицы ПОСЛЕ — все балансы 0!")
show_result(conn.execute("SELECT * FROM clients").fetchall())


## 3. Атака через подбор: WHERE 1=1 (маскировка)

Иногда разработчик «забывает» предикат, иногда — пишет «всегда истинное»
условие (`WHERE 1=1`, `WHERE true`, `WHERE created_at < now()`).
Эффект тот же — обнуление всех строк.


In [ ]:
# Восстановим таблицу
conn = setup_clients_db()


##
# @brief УЯЗВИМАЯ функция с замаскированным «всегда истинным» предикатом.
def zero_balance_BAD_MASKED(conn, client_id: int):
    sql = "UPDATE clients SET balance = 0 WHERE 1 = 1"  # ← синтаксически WHERE есть
    print(f"  Выполняется: {sql}")
    conn.execute(sql)
    conn.commit()


zero_balance_BAD_MASKED(conn, 3)
section("Состояние после маскировки — снова всё нулём")
show_result(conn.execute("SELECT * FROM clients").fetchall())


## 4. Аудитор Phase 1 — правила R002 / R003

В проде:
```python
class UpdateNoWhere(Visitor):
    def visit_UpdateStmt(self, ancestors, node):
        if node.whereClause is None:
            yield Finding("R002-update-no-where", risk_score=9)
        elif is_always_true(node.whereClause):
            yield Finding("R002-update-no-where", risk_score=9, note="маскировка")
```

Здесь — regex-эквивалент.


In [ ]:
##
# @brief Phase 1: ищет UPDATE/DELETE без WHERE или с always-true.
def audit_R002_R003(sql_text):
    findings = []
    is_update = bool(re.match(r"\s*UPDATE\b", sql_text, re.IGNORECASE))
    is_delete = bool(re.match(r"\s*DELETE\b", sql_text, re.IGNORECASE))
    if not (is_update or is_delete):
        return findings

    where_clause = re.search(r"\bWHERE\b(.*?)(?:RETURNING|;|$)", sql_text,
                             re.IGNORECASE | re.DOTALL)

    if where_clause is None:
        # WHERE отсутствует
        findings.append({
            "rule_id":       "R002-update-no-where" if is_update else "R003-delete-no-where",
            "vuln_class":    "DML_NO_WHERE",
            "severity":      "high", "risk_score": 9,
            "message":       f"{'UPDATE' if is_update else 'DELETE'} без WHERE — затрагивает все строки",
            "evidence_refs": ["CWE-1284"],
        })
    else:
        wc = where_clause.group(1).strip().rstrip(";").strip()
        always_true_patterns = [
            r"^\s*1\s*=\s*1\s*$",
            r"^\s*TRUE\s*$",
            r"^\s*'[^']*'\s*=\s*'\\1\\1'\s*$",  # 'x'='x'
        ]
        if any(re.match(p, wc, re.IGNORECASE) for p in always_true_patterns):
            findings.append({
                "rule_id":       "R002-update-no-where" if is_update else "R003-delete-no-where",
                "vuln_class":    "DML_NO_WHERE",
                "severity":      "high", "risk_score": 9,
                "message":       f"WHERE {wc!r} — всегда истинно (маскировка)",
                "evidence_refs": ["CWE-1284"],
            })
    return findings


section("Аудитор по 3 запросам")
for sql in [
    "UPDATE clients SET balance = 0",
    "UPDATE clients SET balance = 0 WHERE 1 = 1",
    "UPDATE clients SET balance = 0 WHERE client_id = ?",  # это безопасный — должно быть тихо
]:
    print(f"\n SQL: {sql}")
    fs = audit_R002_R003(sql)
    if fs:
        for f in fs:
            print_finding(f)
    else:
        print("  ✅ всё ок")


## 5. Безопасная функция


In [ ]:
##
# @brief Безопасная функция: обязательный WHERE по primary key.
# @param client_id  ID конкретного клиента.
def zero_balance_GOOD(conn, client_id: int):
    sql = "UPDATE clients SET balance = 0 WHERE client_id = ?"
    print(f"  SQL: {sql}  | param: client_id = {client_id}")
    conn.execute(sql, (client_id,))
    conn.commit()


conn = setup_clients_db()
section("Безопасный вызов: обнулим клиента #3")
zero_balance_GOOD(conn, 3)
section("Состояние после")
show_result(conn.execute("SELECT * FROM clients").fetchall())


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/04-dml-no-where/README.md](../problems/vulnerabilities/04-dml-no-where/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/04-dml-no-where/solutions.md](../problems/vulnerabilities/04-dml-no-where/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
